# BIRD-SQL — EX & VES theo 6 mức truy vấn (student_club)

Tách 158 câu `student_club` của BIRD dev thành 6 tập theo `sql_category.py`
(1 bảng / inner join / union-intersect-except / outer join / group by / phép chia),
rồi chấm **EX** và **VES** cho 4 model × (có / không knowledge).

Mỗi tập cho ra 2 bảng: `Model | EX_wo_knowledge | EX_w_knowledge` và `Model | VES_wo_knowledge | VES_w_knowledge`.

Chấm bằng chính `execute_model()` / `compute_ves()` của `evaluation.py`, `evaluation_ves.py` gốc
(không chạy 2 script gốc trên từng tập con vì chúng chia cho số câu từng độ khó → lỗi chia 0 khi tập con thiếu độ khó).

Mã nguồn: `app/scripts/sql_category.py`, `app/scripts/ex_ves_by_category.py` (được ghi ra ở 2 cell `%%writefile` bên dưới).

**Chuẩn bị:**
1. Upload `bird-category-eval.zip` (thư mục gốc repo, 8 file `predict_dev.json`) thành Kaggle dataset, ví dụ `bird-category-eval`.
2. *Add Input*: dataset đó **và** dataset BIRD (`bird-sql`, chứa `data/dev.json`, `data/dev_databases`, `llm/src/evaluation*.py`, `llm/data/dev_gold.sql`).
3. Không cần GPU (chỉ chạy SQLite), cần Internet để `pip install func_timeout`. Chạy *Run All*.
4. Tải thư mục `/kaggle/working/ex_ves_by_category_level/` ở tab *Output*.

In [ ]:
!pip install -q func_timeout

In [ ]:
import os

BIRD_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'dev.json' in files and os.path.basename(root) == 'data':
        BIRD_ROOT = os.path.dirname(root)  # thư mục cha của "data/"
        break

assert BIRD_ROOT is not None, "Không tìm thấy bird/data/dev.json trong /kaggle/input — kiểm tra lại dataset đã attach chưa"
print('BIRD_ROOT =', BIRD_ROOT)
print('dev_databases exists:', os.path.exists(f'{BIRD_ROOT}/data/dev_databases'))

import shutil
LOCAL_DB_ROOT = '/kaggle/working/dev_databases'
if not os.path.exists(LOCAL_DB_ROOT):
    shutil.copytree(f'{BIRD_ROOT}/data/dev_databases', LOCAL_DB_ROOT)
print('DBs copied:', len(os.listdir(LOCAL_DB_ROOT)))

LOCAL_SRC = '/kaggle/working/src'
os.makedirs(LOCAL_SRC, exist_ok=True)
shutil.copy(f'{BIRD_ROOT}/llm/src/evaluation.py', LOCAL_SRC)
shutil.copy(f'{BIRD_ROOT}/llm/src/evaluation_ves.py', LOCAL_SRC)
print('evaluation_ves.py đã có --iterate_num:', 'iterate_num' in open(f'{LOCAL_SRC}/evaluation_ves.py').read())

## Bộ phân loại 6 mức (`app/scripts/sql_category.py`)

In [ ]:
%%writefile /kaggle/working/src/sql_category.py
#!/usr/bin/env python3
"""
Phân loại câu SQL theo 6 mức truy vấn (khung trên lớp):

  1. Dễ — truy vấn 1 bảng
  2. Kết 2+ bảng (INNER JOIN, kể cả kết ngầm "FROM a, b")
  3. Phép tập hợp: UNION / INTERSECT / EXCEPT (MINUS)
  4. Kết ngoài: LEFT / RIGHT / FULL OUTER JOIN
  5. Gom nhóm: GROUP BY
  6. Phép chia (relational division): NOT EXISTS lồng NOT EXISTS,
     NOT EXISTS + EXCEPT, hoặc HAVING COUNT(...) = (SELECT COUNT ...)

Một câu SQL có thể mang nhiều nhãn (multi-label). "Mức chính" = nhãn có số lớn nhất.
Câu chỉ dùng nhiều bảng qua truy vấn lồng (WHERE x IN (SELECT ... FROM bảng_khác))
không có JOIN nên không thuộc mức 1–6; được gắn nhãn phụ NESTED và mức chính 0 ("khác").

Dùng như thư viện:
    from sql_category import classify
    classify("SELECT COUNT(*) FROM member GROUP BY position")
    # -> {'tags': ['SINGLE_TABLE', 'GROUP_BY'], 'level': 5, 'tables': ['member'], 'nested': False}

Dùng từ dòng lệnh (gán nhãn tập dev của BIRD, xuất CSV):
    python app/scripts/sql_category.py                      # chỉ student_club
    python app/scripts/sql_category.py --db_id all          # toàn bộ 1.534 câu dev
    python app/scripts/sql_category.py --sql "SELECT ..."   # thử 1 câu
"""
import argparse
import csv
import json
import os
import re
import sys
from collections import Counter

# Windows console (cp1258/cp1252) không in được tiếng Việt -> ép UTF-8
for _s in (sys.stdout, sys.stderr):
    try:
        _s.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

# ---- nhãn ----
SINGLE_TABLE, JOIN, SET_OP, OUTER_JOIN, GROUP_BY, DIVISION = (
    'SINGLE_TABLE', 'JOIN', 'SET_OP', 'OUTER_JOIN', 'GROUP_BY', 'DIVISION')
NESTED = 'NESTED'

LEVEL_OF = {SINGLE_TABLE: 1, JOIN: 2, SET_OP: 3, OUTER_JOIN: 4, GROUP_BY: 5, DIVISION: 6}
LEVEL_NAME = {
    0: 'Khác (nhiều bảng qua truy vấn lồng)',
    1: '1. Dễ - 1 bảng',
    2: '2. Kết bảng (inner join)',
    3: '3. Union / Intersect / Except',
    4: '4. Kết ngoài (outer join)',
    5: '5. Group by',
    6: '6. Phép chia',
}
TAG_ORDER = [SINGLE_TABLE, JOIN, SET_OP, OUTER_JOIN, GROUP_BY, DIVISION, NESTED]

# Từ khoá kết thúc mệnh đề FROM (để phát hiện kết ngầm "FROM a, b")
_FROM_END = {'WHERE', 'GROUP', 'ORDER', 'LIMIT', 'HAVING', 'UNION', 'INTERSECT', 'EXCEPT',
             'MINUS', 'JOIN', 'INNER', 'LEFT', 'RIGHT', 'FULL', 'CROSS', 'NATURAL', 'ON', 'USING',
             'WINDOW'}

_TOKEN_RE = re.compile(r"""
    '(?:[^']|'')*'            # chuỗi 'abc'
  | "(?:[^"]|"")*"            # định danh "abc"
  | `[^`]*`                   # định danh `abc`
  | \[[^\]]*\]                # định danh [abc]
  | [A-Za-z_][A-Za-z0-9_$]*   # từ
  | \d+(?:\.\d+)?             # số
  | <>|!=|<=|>=|\|\||\S       # toán tử / dấu câu
""", re.VERBOSE)


def _tokens(sql: str):
    """Tách token; chuỗi ký tự -> STR, định danh có ngoặc -> giữ tên (chữ thường)."""
    out = []
    for t in _TOKEN_RE.findall(sql):
        if t.startswith("'"):
            out.append(('STR', t))
        elif t[0] in '"`[':
            out.append(('ID', t[1:-1].lower()))
        elif re.match(r'[A-Za-z_]', t):
            out.append(('WORD', t))
        else:
            out.append(('SYM', t))
    return out


def _up(tok):
    return tok[1].upper() if tok[0] == 'WORD' else None


def classify(sql: str) -> dict:
    toks = _tokens(sql or '')
    n = len(toks)
    tags = set()
    tables = set()
    inner_join = outer_join = implicit_join = False
    not_exists = 0
    set_op = False
    nested = False

    for i, tok in enumerate(toks):
        w = _up(tok)
        if w is None:
            if tok == ('SYM', '(') and i + 1 < n and _up(toks[i + 1]) == 'SELECT':
                nested = True
            continue
        nxt = toks[i + 1] if i + 1 < n else None

        if w in ('UNION', 'INTERSECT', 'EXCEPT', 'MINUS'):
            set_op = True
        elif w == 'NOT' and nxt and _up(nxt) == 'EXISTS':
            not_exists += 1
        elif w == 'JOIN':
            # nhìn lùi bỏ qua OUTER để biết LEFT/RIGHT/FULL
            j = i - 1
            if j >= 0 and _up(toks[j]) == 'OUTER':
                j -= 1
            if j >= 0 and _up(toks[j]) in ('LEFT', 'RIGHT', 'FULL'):
                outer_join = True
            else:
                inner_join = True
            if nxt and nxt[0] in ('WORD', 'ID'):
                tables.add(nxt[1].lower())
        elif w == 'FROM':
            if nxt and nxt[0] in ('WORD', 'ID') and _up(nxt) != 'SELECT':
                tables.add(nxt[1].lower())
            # quét tiếp trong cùng mức ngoặc để tìm dấu phẩy => kết ngầm
            depth = 0
            for k in range(i + 1, n):
                t = toks[k]
                if t == ('SYM', '('):
                    depth += 1
                elif t == ('SYM', ')'):
                    if depth == 0:
                        break
                    depth -= 1
                elif depth == 0:
                    if t == ('SYM', ';') or _up(t) in _FROM_END:
                        break
                    if t == ('SYM', ','):
                        implicit_join = True
                        if k + 1 < n and toks[k + 1][0] in ('WORD', 'ID'):
                            tables.add(toks[k + 1][1].lower())

    upper = ' '.join(t[1].upper() if t[0] == 'WORD' else t[1] for t in toks if t[0] != 'STR')
    division = (
        not_exists >= 2
        or (not_exists >= 1 and set_op and 'EXCEPT' in upper)
        or re.search(r'HAVING COUNT \( [^)]*\) = \( SELECT COUNT', upper) is not None
    )

    if inner_join or implicit_join:
        tags.add(JOIN)
    if outer_join:
        tags.add(OUTER_JOIN)
    if set_op:
        tags.add(SET_OP)
    if re.search(r'\bGROUP BY\b', upper):
        tags.add(GROUP_BY)
    if division:
        tags.add(DIVISION)
    if nested:
        tags.add(NESTED)
    if not (inner_join or implicit_join or outer_join) and len(tables) == 1:
        tags.add(SINGLE_TABLE)

    levels = [LEVEL_OF[t] for t in tags if t in LEVEL_OF]
    level = max(levels) if levels else 0
    return {'tags': [t for t in TAG_ORDER if t in tags], 'level': level,
            'tables': sorted(tables), 'nested': nested}


# ---------------------------------------------------------------- CLI

ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..'))
BIRD = os.path.join(ROOT, 'src', 'DAMO-ConvAI', 'bird')
DEV_JSON = os.path.join(BIRD, 'data', 'dev.json')
GOLD_SQL = os.path.join(BIRD, 'llm', 'data', 'dev_gold.sql')


def load_dev_with_gold():
    dev = json.load(open(DEV_JSON, encoding='utf-8'))
    with open(GOLD_SQL, encoding='utf-8') as f:
        gold = [line.rstrip('\n').split('\t')[0] for line in f]
    for i, q in enumerate(dev):
        q['question_id'] = i
        q['gold_sql'] = gold[i]
    return dev


def label_dev(db_id: str):
    dev = load_dev_with_gold()
    if db_id != 'all':
        dev = [q for q in dev if q['db_id'] == db_id]
    rows = []
    for q in dev:
        c = classify(q['gold_sql'])
        rows.append({'question_id': q['question_id'], 'db_id': q['db_id'],
                     'difficulty': q['difficulty'], 'level': c['level'],
                     'level_name': LEVEL_NAME[c['level']], 'tags': '|'.join(c['tags']),
                     'n_tables': len(c['tables']), 'question': q['question'],
                     'gold_sql': q['gold_sql']})
    return rows


def print_summary(rows):
    print(f'\nTổng: {len(rows)} câu')
    print('\nTheo MỨC CHÍNH (mỗi câu đếm 1 lần):')
    lv = Counter(r['level'] for r in rows)
    for k in sorted(LEVEL_NAME):
        print(f'  {LEVEL_NAME[k]:40} {lv.get(k, 0):5}')
    print('\nTheo NHÃN (một câu có thể có nhiều nhãn):')
    tg = Counter(t for r in rows for t in r['tags'].split('|') if t)
    for t in TAG_ORDER:
        print(f'  {t:40} {tg.get(t, 0):5}')


def main():
    ap = argparse.ArgumentParser(description='Gán nhãn 6 mức truy vấn cho gold SQL của BIRD dev')
    ap.add_argument('--db_id', default='student_club', help="tên CSDL, hoặc 'all'")
    ap.add_argument('--sql', help='chỉ phân loại 1 câu SQL rồi thoát')
    ap.add_argument('--out', help='đường dẫn CSV (mặc định docs/pttk/data/sql_category_<db>.csv)')
    args = ap.parse_args()

    if args.sql:
        c = classify(args.sql)
        print(json.dumps({**c, 'level_name': LEVEL_NAME[c['level']]}, ensure_ascii=False, indent=2))
        return

    rows = label_dev(args.db_id)
    out = args.out or os.path.join(ROOT, 'docs', 'pttk', 'data', f'sql_category_{args.db_id}.csv')
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, 'w', newline='', encoding='utf-8-sig') as f:   # utf-8-sig: Excel đọc đúng tiếng Việt
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print_summary(rows)
    print(f'\n-> {out}')


if __name__ == '__main__':
    main()

## Script chấm EX & VES theo mức (`app/scripts/ex_ves_by_category.py`)

In [ ]:
%%writefile /kaggle/working/src/ex_ves_by_category.py
#!/usr/bin/env python3
"""
EX và VES theo 6 mức truy vấn (sql_category.py) cho từng model, có / không knowledge.

Mỗi mức (tập câu hỏi) cho ra 2 bảng:
  EX : Model | EX_wo_knowledge  | EX_w_knowledge
  VES: Model | VES_wo_knowledge | VES_w_knowledge

Chấm bằng chính bộ chấm gốc của BIRD (llm/src/evaluation.py, evaluation_ves.py):
  - package_sqls()   đọc predict_dev.json / dev_gold.sql
  - execute_model()  EX: so tập kết quả;  VES: tỉ lệ thời gian gold/pred (iterate_num lần)
  - compute_ves()    VES = trung bình sqrt(time_ratio) * 100
Không gọi thẳng 2 script gốc trên từng tập con vì chúng chia cho số câu
simple/moderate/challenging -> tập con thiếu một độ khó sẽ lỗi chia cho 0.

Chạy trên Kaggle (xem ex-ves-by-category-kaggle.ipynb) hoặc local:
  python app/scripts/ex_ves_by_category.py                         # student_club, local
  python app/scripts/ex_ves_by_category.py --group tag             # theo nhãn (một câu nhiều nhãn)
  python app/scripts/ex_ves_by_category.py --skip_ves              # chỉ EX (nhanh)
  python ex_ves_by_category.py --bird_root /kaggle/input/.../bird \\
      --eval_src /kaggle/working/src --db_root_path /kaggle/working/dev_databases \\
      --out_dir /kaggle/working/ex_ves_by_category

Xuất (trong --out_dir):
  per_question_<db>.csv         kết quả từng câu × từng lượt (res EX, time_ratio VES)
  ex_ves_by_category_<db>.csv   bảng dài: mức, model, n, 4 cột EX/VES
  ex_ves_by_category_<db>.xlsx  mỗi mức 2 sheet (EX, VES) — nếu có openpyxl
"""
import argparse
import json
import multiprocessing as mp
import os
import sys

import pandas as pd

# Windows console (cp1258/cp1252) không in được tiếng Việt -> ép UTF-8
for _s in (sys.stdout, sys.stderr):
    try:
        _s.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from sql_category import LEVEL_NAME, TAG_ORDER, classify  # noqa: E402

ROOT = os.path.abspath(os.path.join(HERE, '..', '..'))
LOCAL_BIRD = os.path.join(ROOT, 'src', 'DAMO-ConvAI', 'bird')

# model hiển thị -> (thư mục w/ knowledge, thư mục w/o knowledge) — giống evaluation-and-ves-local.ipynb
MODEL_FOLDERS = {
    'DeepSeek-Coder-V2': ('ollama_deepseekcoder_output', 'ollama_deepseekcoder_0_output'),
    'Gemma3':            ('ollama_gemma3_output',        'ollama_gemma3_0_output'),
    'Qwen2.5-Coder':     ('ollama_gwen25_output',        'ollama_gwen25_0_output'),
    'GPT-3.5-Turbo':     ('openai_output',               'openai_0_output'),
}

TAG_NAME = {
    'SINGLE_TABLE': '1. Đơn bảng',
    'JOIN': '2. Join',
    'SET_OP': '3. Union / Intersect / Except',
    'OUTER_JOIN': '4. Outer join',
    'GROUP_BY': '5. Group by',
    'DIVISION': '6. Phép chia',
}

COLS = ['EX_wo_knowledge', 'EX_w_knowledge', 'VES_wo_knowledge', 'VES_w_knowledge']


def find_bird_root():
    """Kaggle: tìm thư mục cha của data/dev.json trong /kaggle/input; local: src/DAMO-ConvAI/bird."""
    if os.path.isdir('/kaggle/input'):
        for root, _dirs, files in os.walk('/kaggle/input'):
            if 'dev.json' in files and os.path.basename(root) == 'data':
                return os.path.dirname(root)
    return LOCAL_BIRD


def categories(questions, group):
    """[(key, tên, [question_id...])] — 'level': 6 tập rời nhau theo mức chính; 'tag': theo nhãn."""
    if group == 'level':
        cats = [(str(lv), LEVEL_NAME[lv], [q['question_id'] for q in questions if q['level'] == lv])
                for lv in range(1, 7)]
        other = [q['question_id'] for q in questions if q['level'] == 0]
        if other:
            cats.append(('0', LEVEL_NAME[0], other))
    else:
        cats = [(t, TAG_NAME[t], [q['question_id'] for q in questions if t in q['tags']])
                for t in TAG_ORDER if t in TAG_NAME]
    cats.append(('ALL', 'Tổng', [q['question_id'] for q in questions]))
    return cats


def score_run(evaluation, evaluation_ves, pred_dir, idxs, gold_sqls, args):
    """Chấm 1 lượt (1 thư mục predict_dev.json) trên các câu idxs. Trả {idx: (res, time_ratio)}."""
    preds, pred_dbs = evaluation.package_sqls(pred_dir + '/', args.db_root_path, mode='gpt', data_mode='dev')
    with mp.Pool(processes=args.num_cpus) as pool:
        ex = pool.starmap(evaluation.execute_model,
                          [(preds[i], gold_sqls[i], pred_dbs[i], i, args.meta_time_out) for i in idxs])
        if args.skip_ves:
            ves = [{'sql_idx': i, 'time_ratio': None} for i in idxs]
        else:
            ves = pool.starmap(evaluation_ves.execute_model,
                               [(preds[i], gold_sqls[i], pred_dbs[i], i, args.iterate_num, args.meta_time_out)
                                for i in idxs])
    ratio = {r['sql_idx']: r['time_ratio'] for r in ves}
    return {r['sql_idx']: (r['res'], ratio[r['sql_idx']]) for r in ex}


def main():
    ap = argparse.ArgumentParser(description='EX & VES theo 6 mức truy vấn, có / không knowledge')
    ap.add_argument('--bird_root', help='thư mục chứa data/ và llm/ (mặc định tự tìm)')
    ap.add_argument('--eval_src', help='thư mục chứa evaluation.py, evaluation_ves.py (mặc định <bird>/llm/src)')
    ap.add_argument('--db_root_path', help='thư mục dev_databases (mặc định <bird>/data/dev_databases)')
    ap.add_argument('--exp_dir', help='thư mục exp_result (mặc định <bird>/llm/exp_result)')
    ap.add_argument('--db_id', default='student_club', help="tên CSDL, hoặc 'all'")
    ap.add_argument('--group', choices=['level', 'tag'], default='level',
                    help='level: 6 tập rời nhau theo mức chính (mặc định); tag: theo nhãn')
    ap.add_argument('--num_cpus', type=int, default=4)
    ap.add_argument('--meta_time_out', type=float, default=30.0)
    ap.add_argument('--iterate_num', type=int, default=5, help='số lần chạy lại SQL khi đo VES (paper: 5)')
    ap.add_argument('--skip_ves', action='store_true', help='chỉ chấm EX')
    ap.add_argument('--out_dir', help='mặc định docs/pttk/data')
    args = ap.parse_args()

    bird = args.bird_root or find_bird_root()
    eval_src = args.eval_src or os.path.join(bird, 'llm', 'src')
    args.db_root_path = os.path.join(args.db_root_path or os.path.join(bird, 'data', 'dev_databases'), '')
    exp_dir = args.exp_dir or os.path.join(bird, 'llm', 'exp_result')
    out_dir = args.out_dir or os.path.join(ROOT, 'docs', 'pttk', 'data')
    os.makedirs(out_dir, exist_ok=True)
    print('BIRD_ROOT =', bird)

    sys.path.insert(0, eval_src)
    import evaluation       # noqa: E402  (bộ chấm gốc)
    import evaluation_ves   # noqa: E402

    dev = json.load(open(os.path.join(bird, 'data', 'dev.json'), encoding='utf-8'))
    gold_sqls, _ = evaluation.package_sqls(os.path.join(bird, 'llm', 'data', ''), args.db_root_path,
                                           mode='gt', data_mode='dev')
    questions = []
    for i, q in enumerate(dev):
        if args.db_id != 'all' and q['db_id'] != args.db_id:
            continue
        c = classify(gold_sqls[i])
        questions.append({'question_id': i, 'difficulty': q['difficulty'], 'level': c['level'], 'tags': c['tags']})
    if not questions:
        sys.exit(f'Không có câu nào với db_id={args.db_id}')
    idxs = [q['question_id'] for q in questions]
    cats = categories(questions, args.group)
    print(f'{args.db_id}: {len(questions)} câu; phân loại theo {args.group}:')
    for key, name, ids in cats:
        print(f'  {name:40} {len(ids):5}')

    # ---- chấm từng lượt: model × (có / không knowledge) ----
    results = {}  # (model, 'w'|'wo') -> {idx: (res, time_ratio)}
    for model, (kg_dir, nokg_dir) in MODEL_FOLDERS.items():
        for label, d in [('wo', nokg_dir), ('w', kg_dir)]:
            pred_dir = os.path.join(exp_dir, d)
            if not os.path.exists(os.path.join(pred_dir, 'predict_dev.json')):
                print(f'  bỏ qua {model} ({label} knowledge): không có {pred_dir}/predict_dev.json')
                continue
            print(f'\n===== {model} ({label} knowledge): {d} =====')
            results[(model, label)] = score_run(evaluation, evaluation_ves, pred_dir, idxs, gold_sqls, args)
            r = results[(model, label)]
            print(f'  EX = {100 * sum(v[0] for v in r.values()) / len(r):.2f}')

    if not results:
        sys.exit('Không có lượt nào để chấm.')

    # ---- từng câu ----
    qmap = {q['question_id']: q for q in questions}
    per_q = pd.DataFrame([
        {'question_id': i, 'difficulty': qmap[i]['difficulty'], 'level': qmap[i]['level'],
         'tags': '|'.join(qmap[i]['tags']), 'model': model, 'knowledge': label,
         'ex': res, 'time_ratio': ratio}
        for (model, label), r in results.items() for i, (res, ratio) in r.items()])
    per_q.to_csv(os.path.join(out_dir, f'per_question_{args.db_id}.csv'), index=False, encoding='utf-8-sig')

    # ---- gom theo mức: EX = % đúng; VES = compute_ves() gốc ----
    rows = []
    for key, name, ids in cats:
        for model in MODEL_FOLDERS:
            row = {'category': key, 'category_name': name, 'n': len(ids), 'model': model}
            for label in ('wo', 'w'):
                r = results.get((model, label))
                ex = ves = None
                if r is not None and ids:
                    ex = round(100 * sum(r[i][0] for i in ids) / len(ids), 2)
                    if not args.skip_ves:
                        ves = round(evaluation_ves.compute_ves([{'time_ratio': r[i][1]} for i in ids]), 2)
                row[f'EX_{label}_knowledge'] = ex
                row[f'VES_{label}_knowledge'] = ves
            rows.append(row)
    long = pd.DataFrame(rows)[['category', 'category_name', 'n', 'model'] + COLS]
    out_csv = os.path.join(out_dir, f'ex_ves_by_category_{args.db_id}.csv')
    long.to_csv(out_csv, index=False, encoding='utf-8-sig')

    # ---- in bảng: mỗi mức 1 bảng EX + 1 bảng VES ----
    sheets = {}
    for key, name, ids in cats:
        part = long[long['category'] == key].set_index('model')
        ex_tbl = part[['EX_wo_knowledge', 'EX_w_knowledge']]
        ves_tbl = part[['VES_wo_knowledge', 'VES_w_knowledge']]
        sheets[f'{key}_EX'], sheets[f'{key}_VES'] = ex_tbl, ves_tbl
        print(f'\n######## {name} — {len(ids)} câu ########')
        if not ids:
            print('  (không có câu nào thuộc tập này)')
            continue
        print('EX (%)\n' + ex_tbl.to_string(na_rep='-'))
        if not args.skip_ves:
            print('VES\n' + ves_tbl.to_string(na_rep='-'))

    try:
        out_xlsx = os.path.join(out_dir, f'ex_ves_by_category_{args.db_id}.xlsx')
        with pd.ExcelWriter(out_xlsx) as w:
            for sheet, tbl in sheets.items():
                tbl.to_excel(w, sheet_name=sheet[:31])
        print(f'\n-> {out_xlsx}')
    except ImportError:
        print('\n(không có openpyxl -> bỏ qua file .xlsx)')
    print(f'-> {out_csv}')


if __name__ == '__main__':
    main()

In [ ]:
# Tìm thư mục chứa 8 thư mục kết quả: ưu tiên dataset bird-category-eval, không có thì dùng exp_result của dataset BIRD
import glob

hits = glob.glob('/kaggle/input/**/openai_0_output/predict_dev.json', recursive=True)
hits.sort(key=lambda h: '/llm/exp_result/' in h)
EXP_DIR = os.path.dirname(os.path.dirname(hits[0])) if hits else f'{BIRD_ROOT}/llm/exp_result'
print('EXP_DIR =', EXP_DIR)

# model hiển thị -> (thư mục w/ knowledge, thư mục w/o knowledge) — phải khớp MODEL_FOLDERS trong script
MODEL_FOLDERS = {
    "DeepSeek-Coder-V2": ("ollama_deepseekcoder_output", "ollama_deepseekcoder_0_output"),
    "Gemma3":            ("ollama_gemma3_output",        "ollama_gemma3_0_output"),
    "Qwen2.5-Coder":     ("ollama_gwen25_output",         "ollama_gwen25_0_output"),
    "GPT-3.5-Turbo":     ("openai_output",                "openai_0_output"),
}
for name, (kg_dir, nokg_dir) in MODEL_FOLDERS.items():
    for label, d in [("w/ knowledge", kg_dir), ("w/o knowledge", nokg_dir)]:
        p = f'{EXP_DIR}/{d}/predict_dev.json'
        print(f"{name:20} {label:15} -> {'OK' if os.path.exists(p) else 'MISSING'}  {p}")

In [ ]:
import subprocess, sys

DB_ID = 'student_club'
GROUP = 'level'      # 'level': 6 tập rời nhau theo mức chính | 'tag': theo nhãn (một câu có thể nhiều nhãn)
ITERATE_NUM = 5      # như paper; phải >= 2 (=1 thì clean_abnormal của bộ chấm gốc loại hết mẫu -> VES = 0)
OUT_DIR = f'/kaggle/working/ex_ves_by_category_{GROUP}'

cmd = [
    sys.executable, f'{LOCAL_SRC}/ex_ves_by_category.py',
    '--bird_root', BIRD_ROOT,
    '--eval_src', LOCAL_SRC,
    '--db_root_path', LOCAL_DB_ROOT,
    '--exp_dir', EXP_DIR,
    '--db_id', DB_ID,
    '--group', GROUP,
    '--num_cpus', '4',
    '--meta_time_out', '30.0',
    '--iterate_num', str(ITERATE_NUM),
    '--out_dir', OUT_DIR,
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])

## Bảng EX và VES cho từng tập câu hỏi

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

long = pd.read_csv(f'{OUT_DIR}/ex_ves_by_category_{DB_ID}.csv', dtype={'category': str})
for (key, name, n), part in long.groupby(['category', 'category_name', 'n'], sort=False):
    display(Markdown(f'### {name} — {n} câu'))
    if n == 0:
        display(Markdown('_Không có câu nào thuộc tập này trong ' + DB_ID + '._'))
        continue
    part = part.set_index('model')
    display(Markdown('**EX (%)**'))
    display(part[['EX_wo_knowledge', 'EX_w_knowledge']])
    display(Markdown('**VES**'))
    display(part[['VES_wo_knowledge', 'VES_w_knowledge']])

In [ ]:
# Bảng tổng hợp gọn: hàng = (tập, model), cột = 4 chỉ số — tiện dán vào báo cáo
summary = long[long['n'] > 0].set_index(['category_name', 'model'])[
    ['n', 'EX_wo_knowledge', 'EX_w_knowledge', 'VES_wo_knowledge', 'VES_w_knowledge']]
summary.to_csv(f'{OUT_DIR}/summary_ex_ves_by_category_{DB_ID}.csv')
print('Files:', os.listdir(OUT_DIR))
summary